# 11:11 · Wan 2.2 anime test (free Kaggle GPU)

Animates two still frames from 11:11 (Kabir and Mira on the street at dusk) with **FastWan 2.2 TI2V-5B**, the open-source Wan 2.2 model distilled to 4 steps. It prints how long each clip takes, so we know how many clips a week of free GPU buys.

**Before running:** in the right-hand panel set **Accelerator → GPU T4 x2** (or GPU T4) and turn **Internet → On** (needs a phone-verified account). Then **Run All**.

The first run downloads about 20–30 GB of model files, which takes 10–20 minutes. The clips land in **Output** (`/kaggle/working`) as `kabir.mp4` and `mira.mp4`.

In [ ]:
# 1. Check the machine
import os, shutil, subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
print('RAM GB:', round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9, 1))
for p in ['/kaggle/tmp', '/tmp', '/kaggle/working']:
    if os.path.exists(p): print(p, 'free GB:', round(shutil.disk_usage(p).free / 1e9, 1))
# keep the big model download out of /kaggle/working (that folder is capped at 20 GB)
CACHE = '/kaggle/tmp/hf' if os.path.exists('/kaggle/tmp') else '/tmp/hf'
os.environ['HF_HOME'] = CACHE
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
print('model cache:', CACHE)

In [ ]:
# 2. Install the libraries (about 2 minutes)
!pip install -q -U "diffusers>=0.35" "transformers>=4.49" accelerate ftfy sentencepiece imageio imageio-ffmpeg hf_transfer

In [ ]:
# 3. The settings for this test
MODEL_ID = 'FastVideo/FastWan2.2-TI2V-5B-FullAttn-Diffusers'
WIDTH, HEIGHT = 480, 864      # vertical; both must be multiples of 32
FRAMES = 73                   # 3 seconds at 24 fps (must be 4k + 1)
STEPS = 4                     # FastWan is distilled for 3-5 steps
DTYPE = 'float16'             # the T4 has no fast bfloat16; if the clip comes out black or grey, see the last cell
SEED = 11
NEG = ('static, still picture, blurred details, low quality, worst quality, JPEG artifacts, deformed, disfigured, '
       'extra fingers, poorly drawn hands, poorly drawn face, fused fingers, watermark, text, subtitles, '
       '3D render, realistic, photorealistic, glossy')
SHOTS = [
    dict(name='kabir', image='https://raw.githubusercontent.com/PriyankaKondapaneni/siri.ai/claude/anime-rig-test/shorts/eleven/kaggle/test-kabir-street.jpg',
         prompt='Anime animation in a flat, hand-drawn webtoon style. A young man in a charcoal coat stands on a city street at sunset. '
                'He slowly turns his head toward the camera, blinks, and smiles warmly. His messy black hair moves gently in the breeze. '
                'Warm golden light, a slow camera push-in. Keep the same character design, flat colours and thin ink outlines.'),
    dict(name='mira', image='https://raw.githubusercontent.com/PriyankaKondapaneni/siri.ai/claude/anime-rig-test/shorts/eleven/kaggle/test-mira-street.jpg',
         prompt='Anime animation in a flat, hand-drawn webtoon style. A young woman with long wavy black hair, a red scarf and a blue dress '
                'stands on a city street at sunset, holding her bag strap. She looks around, then smiles shyly and tucks her hair behind her ear. '
                'Her hair and scarf move in the breeze. Warm golden light, gentle camera movement. Keep the same character design, flat colours and thin ink outlines.'),
]

In [ ]:
# 4. Fetch the two starting frames and crop them to the video shape
import requests, io
from PIL import Image
def fetch(url):
    im = Image.open(io.BytesIO(requests.get(url, timeout=60).content)).convert('RGB')
    w, h = im.size; target = WIDTH / HEIGHT
    if w / h > target: nw = int(h * target); im = im.crop(((w - nw) // 2, 0, (w - nw) // 2 + nw, h))
    else: nh = int(w / target); im = im.crop((0, (h - nh) // 2, w, (h - nh) // 2 + nh))
    return im.resize((WIDTH, HEIGHT), Image.LANCZOS)
for s in SHOTS: s['img'] = fetch(s['image'])
display(SHOTS[0]['img'].resize((240, 432)), SHOTS[1]['img'].resize((240, 432)))

In [ ]:
# 5. Read the prompts with the text encoder on the CPU (bfloat16 works there; on the T4 it would be slow or unstable),
#    then free its memory before the video model loads
import torch, time, gc
from transformers import AutoTokenizer, UMT5EncoderModel
t0 = time.time()
tok = AutoTokenizer.from_pretrained(MODEL_ID, subfolder='tokenizer')
te = UMT5EncoderModel.from_pretrained(MODEL_ID, subfolder='text_encoder', torch_dtype=torch.bfloat16)
te.eval()
def encode(text, L=512):
    t = tok([text], padding='max_length', max_length=L, truncation=True, add_special_tokens=True, return_attention_mask=True, return_tensors='pt')
    with torch.no_grad():
        e = te(t.input_ids, t.attention_mask).last_hidden_state
    n = int(t.attention_mask.sum())
    out = torch.zeros_like(e); out[:, :n] = e[:, :n]   # the same zero padding Wan uses
    return out.float()
for s in SHOTS:
    s['emb'] = encode(s['prompt'])
NEG_EMB = encode(NEG)
del te; gc.collect()
print(f'prompts encoded in {time.time() - t0:.0f} s')

In [ ]:
# 6. Load the video model onto the GPU
from diffusers import AutoencoderKLWan, WanImageToVideoPipeline, UniPCMultistepScheduler
t0 = time.time()
dt = getattr(torch, DTYPE)
vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder='vae', torch_dtype=torch.float32)
pipe = WanImageToVideoPipeline.from_pretrained(MODEL_ID, vae=vae, text_encoder=None, tokenizer=None, torch_dtype=dt)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=8.0)
pipe.to('cuda')
try: pipe.vae.enable_tiling()
except Exception as e: print('no VAE tiling:', e)
print(f'model loaded in {time.time() - t0:.0f} s')
print('GPU memory used GB:', round(torch.cuda.memory_allocated() / 1e9, 1))

In [ ]:
# 7. Make the clips and time them
import numpy as np
from diffusers.utils import export_to_video
results = []
for s in SHOTS:
    torch.cuda.synchronize(); t0 = time.time()
    with torch.inference_mode():
        frames = pipe(image=s['img'], prompt_embeds=s['emb'].to('cuda', dt), negative_prompt_embeds=NEG_EMB.to('cuda', dt),
                      height=HEIGHT, width=WIDTH, num_frames=FRAMES, num_inference_steps=STEPS, guidance_scale=0.0,
                      generator=torch.Generator(device='cuda').manual_seed(SEED)).frames[0]
    torch.cuda.synchronize(); took = time.time() - t0
    arr = np.asarray(frames)
    ok = np.isfinite(arr).all() and arr.std() > 0.02
    path = f"/kaggle/working/{s['name']}.mp4"
    export_to_video(frames, path, fps=24)
    results.append((s['name'], took, ok))
    print(f"{s['name']}: {took:.0f} s for {FRAMES / 24:.1f} s of video   {'OK' if ok else 'LOOKS BROKEN (black/grey) - see last cell'}")
avg = sum(r[1] for r in results) / len(results)
print(f'\nAbout {30 * 3600 / avg:.0f} clips like this per week of free Kaggle GPU (30 h).')

In [ ]:
# 8. Watch them here
from IPython.display import Video, display
for s in SHOTS: display(Video(f"/kaggle/working/{s['name']}.mp4", embed=True, width=270))

### If a clip came out black, grey or noisy
That means the model overflowed in float16 on the T4. Change `DTYPE = 'float16'` in cell 3 to `DTYPE = 'float32'`, add the line `pipe.enable_sequential_cpu_offload()` instead of `pipe.to('cuda')` in cell 6, and run all again. It will be much slower, but it shows whether the problem is precision.

### What to send back
The two clips from **Output**, plus the timing lines printed by cell 7.